In [1]:
import numpy as np
import pyscf
import pyscf.gto
import pyscf.scf
import pyscf.mcscf
import pyscf.cc
import pyscf.ci
from pyscf import lib
from scipy.sparse.linalg import LinearOperator
from ffsim.optimize import minimize_linear_method

from xquces.igcr2 import IGCR2SpinBalancedParameterization
from xquces.hamiltonians import MolecularHamiltonianLinearOperator
from xquces.states import hartree_fock_state
from xquces.ucj.init import UCJBalancedDFSeed


# **Optimization with the linear method**

In [2]:
n_f = 0
R = 1.0
basis = "cc-pvdz"

def linear_operator_from_xquces_hamiltonian(ham):
    dim = hartree_fock_state(ham.norb, ham.nelec).size

    def matvec(v):
        v = np.asarray(v, dtype=np.complex128).reshape(-1)
        return ham.matvec(v) + ham.ecore * v

    def matmat(vs):
        vs = np.asarray(vs, dtype=np.complex128)
        return np.column_stack([matvec(vs[:, j]) for j in range(vs.shape[1])])

    return LinearOperator(
        shape=(dim, dim),
        matvec=matvec,
        matmat=matmat,
        dtype=np.complex128,
    )


mol = pyscf.gto.Mole()
mol.build(
    atom=[("H", (-0.5 * R, 0, 0)), ("H", (0.5 * R, 0, 0))],
    basis=basis,
    symmetry="Dooh",
    verbose=0,
)

scf = pyscf.scf.RHF(mol)
scf.kernel()

active_space = list(range(n_f, mol.nao_nr()))
norb = len(active_space)
nelectron_cas = int(round(sum(scf.mo_occ[active_space])))
n_alpha = (nelectron_cas + mol.spin) // 2
n_beta = (nelectron_cas - mol.spin) // 2
nelec = (n_alpha, n_beta)

cas = pyscf.mcscf.RCASCI(scf, ncas=norb, nelecas=nelec)
mo_coeff = cas.sort_mo(active_space, base=0)
active_mo_coeff = np.asarray(
    mo_coeff[:, cas.ncore : cas.ncore + norb],
    dtype=np.complex128,
)

cisd = pyscf.ci.RCISD(
    scf, frozen=[i for i in range(mol.nao_nr()) if i not in active_space]
)
cisd.kernel()

ccsd = pyscf.cc.RCCSD(
    scf, frozen=[i for i in range(mol.nao_nr()) if i not in active_space]
)
ccsd.kernel(
    t1=ccsd.t1,
    t2=ccsd.t2,
)

cas.fix_spin_(ss=0)
cas.kernel(mo_coeff=mo_coeff)

ham_xq = MolecularHamiltonianLinearOperator.from_scf(scf, active_space=active_space)
H = linear_operator_from_xquces_hamiltonian(ham_xq)
Phi0 = hartree_fock_state(norb, nelec)

ucj_seed = UCJBalancedDFSeed(
    t2=ccsd.t2,
    t1=ccsd.t1,
    n_reps=1,
).build_ansatz()

igcr2_param = IGCR2SpinBalancedParameterization(
    norb=norb,
    nocc=n_alpha,
)

x0_seed = igcr2_param.parameters_from_ucj_ansatz(ucj_seed)

print("Number of parameters:", len(x0_seed), flush=True)

psi_seed = igcr2_param.ansatz_from_parameters(x0_seed).apply(Phi0, nelec=nelec, copy=True)
E_iGCR2_seed = ham_xq.expectation(psi_seed)

def params_to_vec(x):
    return igcr2_param.ansatz_from_parameters(x).apply(Phi0, nelec=nelec, copy=True)

it_counter = {"k": 0}

def callback(intermediate_result):
    it_counter["k"] += 1
    energy = float(intermediate_result.fun)

    jac = getattr(intermediate_result, "jac", None)
    if jac is None:
        gmax = np.nan
    else:
        gmax = float(np.max(np.abs(jac)))

    cond = float(getattr(intermediate_result, "cond_S", np.nan))
    regularization = float(getattr(intermediate_result, "regularization", np.nan))
    variation = float(getattr(intermediate_result, "variation", np.nan))

    print(
        f"Iter {it_counter['k']}: "
        f"E = {energy:.12f}, "
        f"gmax = {gmax:.2e}, "
        f"cond_S = {cond:.2e}, "
        f"reg = {regularization:.2e}, "
        f"var = {variation:.2e}",
        flush=True,
    )



result = minimize_linear_method(
    params_to_vec,
    H,
    x0=x0_seed,
    maxiter=100,
    gtol=1e-6,
    ftol=1e-12,
    callback=callback,
)

x_prev1 = result.x.copy()
prev_mol = mol
prev_active_mo_coeff = active_mo_coeff.copy()
prev_igcr2_param = igcr2_param
E_iGCR2_opt = float(result.fun)

E_HF = scf.e_tot
E_FCI = cas.e_tot

print("Final results:")
print(f"E(HF) = {E_HF:.12f}")
print(f"E(FCI) = {E_FCI:.12f}")
print(f"E(iGCR2 seed) = {E_iGCR2_seed:.12f}")
print(f"E(iGCR2) = {E_iGCR2_opt:.12f}")
print(f"Correlation energy captured: {(E_iGCR2_opt - E_HF) / (E_FCI - E_HF) * 100:.2f}%")

Number of parameters: 163
Iter 1: E = -1.127197900604, gmax = 7.54e-04, cond_S = 3.70e+21, reg = 2.98e-05, var = 9.81e-01
Iter 2: E = -1.127239114232, gmax = 7.35e-03, cond_S = 3.89e+19, reg = 4.72e-03, var = 9.84e-01
Iter 3: E = -1.128668261750, gmax = 5.77e-02, cond_S = 1.97e+19, reg = 1.83e-03, var = 6.26e-01
Iter 4: E = -1.132418518676, gmax = 2.18e-02, cond_S = 3.57e+19, reg = 2.69e-03, var = 6.26e-01
Iter 5: E = -1.135445490315, gmax = 3.63e-02, cond_S = 4.55e+21, reg = 1.44e-03, var = 9.99e-01
Iter 6: E = -1.137406378955, gmax = 3.93e-02, cond_S = 4.41e+18, reg = 6.32e-03, var = 9.99e-01
Iter 7: E = -1.139107270514, gmax = 1.07e-02, cond_S = 6.62e+18, reg = 1.94e-04, var = 9.99e-01
Iter 8: E = -1.139490821237, gmax = 2.51e-02, cond_S = 1.88e+19, reg = 5.31e-04, var = 9.99e-01
Iter 9: E = -1.139858579075, gmax = 9.25e-03, cond_S = 1.20e+18, reg = 7.49e-04, var = 9.99e-01
Iter 10: E = -1.140056381455, gmax = 2.60e-03, cond_S = 2.03e+18, reg = 5.50e-05, var = 9.99e-01
Iter 11: E = 